# Ai Agent Autopsy - 03 Open AI Agents SDK

This shows a simple example of using the Open AI SDK to create a sub-agent by defining an agent as a tool that can be used by another agent.

In [10]:
from agents import Agent, Runner, trace, function_tool
from pydantic import BaseModel
from pathlib import Path

NOTE: For the sake of simplcity this example uses both the Open AI models and the Open AI agents SDK.

In reality the SDK will work with any LLM endpoint that supports the Open AI standard.

To run this example you will need your environment configured with an Open AI API key.

In [ ]:
%%bash

echo $OPENAI_API_KEY

## Debugging Agent with a Planning Sub-Agent Tool

In [7]:
class PlanningOutput(BaseModel):
    """
    Structure to return from the planning agent.
    Include a summary of what is identified as the problem in the stack trace.
    And a list of source code files with complete paths to be read in and analysed.
    """
    summary: str
    file_paths: list[str]

In [9]:
planner_instructions = """
You a software developer whose role is to plan debugging activity.
You will be provided with a stack trace from a failing program you should determine 
which files should be read into memory for analysis by another agent.
"""

planner_agent = Agent(
    name="Planning Agent",
    instructions=planner_instructions,
    model="gpt-4o-mini",
    output_type=PlanningOutput
)

planner_tool = planner_agent.as_tool(
    tool_name="planner_agent", 
    tool_description="Analyse a stack trace and plan which files should be read in for analysis."
)

In [17]:
@function_tool
def read_file(file_path: str):
    """ Read in a file from disk for analysis """
    print(f"Trying to read:{file_path}")
    if Path(file_path).is_file():
        with open(file_path, 'r') as file:
            content = file.read()
        return content
    else:
        return f"File {file_path} not found"

In [18]:
tools = [planner_tool, read_file]

## Create the Analysis Agent

The final step is to create an agent that will perform the analysis for us using the tools.

In [13]:
instructions = """
You are a Software Debugging Enginer. 
Your goal is to determine the likely source of an error in some software from the provided stack trace.

Follow these steps carefully:
1. Use the planning tool to determine which files to read and gain some insight into the potential problem.
2. Load the suggested file with the read_file tool
3. Analyse the code in light of the stack trace to determine the most likely issue.
"""

debugger = Agent(name="Code Debugging Agent", instructions=instructions, tools=tools, model="gpt-4o-mini")

## Example data 

I created a trivial example for illustrative purposes

-- You should apply this to your own examples

In [15]:
stack_trace = """
Traceback (most recent call last):
  File "/Users/john/Projects/data-science-first/extras/bad_main.py", line 3, in <module>
    func_a()
  File "/Users/john/Projects/data-science-first/extras/bad_module.py", line 2, in func_a
    func_b()
  File "/Users/john/Projects/data-science-first/extras/bad_module.py", line 5, in func_b
    1 / 0
    ~~^~~
ZeroDivisionError: division by zero
"""

In [19]:

result = await Runner.run(debugger, stack_trace)

print(result)

RunResult:
- Last agent: Agent(name="Code Debugging Agent", ...)
- Final output (str):
    ### Analysis
    
    1. **Stack Trace Review**:
       - The error is a `ZeroDivisionError`, which indicates that there was an attempt to divide by zero in `func_b`.
       - The call sequence shows:
         - `bad_main.py` calls `func_a`.
         - `func_a` in `bad_module.py` calls `func_b`.
         - The actual division error occurs in `func_b`.
    
    2. **Code Examination**:
       - **`bad_main.py`**:
         ```python
         from bad_module import func_a
    
         func_a()
         ```
       - **`bad_module.py`**:
         ```python
         def func_a():
             func_b()
    
         def func_b():
             1 / 0
         ```
    
    ### Conclusion
    The source of the error is clear: in `func_b`, the line `1 / 0` attempts to divide by zero, which raises the `ZeroDivisionError`. 
    
    ### Suggested Fix
    To fix the error, modify `func_b` to avoid dividing by 

In [20]:
print(result.raw_responses)

[ModelResponse(output=[ResponseFunctionToolCall(arguments='{"input":"Traceback (most recent call last):\\n  File \\"/Users/john/Projects/data-science-first/extras/bad_main.py\\", line 3, in <module>\\n    func_a()\\n  File \\"/Users/john/Projects/data-science-first/extras/bad_module.py\\", line 2, in func_a\\n    func_b()\\n  File \\"/Users/john/Projects/data-science-first/extras/bad_module.py\\", line 5, in func_b\\n    1 / 0\\n    ~~^~~\\nZeroDivisionError: division by zero\\n\\nThis error indicates that there was an attempt to divide by zero in the function func_b of bad_module.py."}', call_id='call_CWS3b0HQBHWxl7xPUhljI5gZ', name='planner_agent', type='function_call', id='fc_0e28a75ef76da0160069dcadc1e0fc8194a9ee18c103a2b50c', namespace=None, status='completed')], usage=Usage(requests=1, input_tokens=307, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=164, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=471, request_usage_entries